In [1]:
import numpy as np
from qdk_chemistry.data import Element, Structure, MajoranaMapping
from qdk_chemistry.algorithms import create
from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)

coords_ang = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.740848],
    ]
)
structure = Structure(coords_ang, [Element.H, Element.H])

scf = create("scf_solver")
scf_energy, wavefunction = scf.run(
    structure, charge=0, spin_multiplicity=1, basis_or_guess="sto-3g"
)

# 2) Electronic Hamiltonian
ham_constructor = create("hamiltonian_constructor")
hamiltonian = ham_constructor.run(wavefunction.get_orbitals())

# 3) Qubit Hamiltonian
mapper = create("qubit_mapper")
n_spin_orbitals = hamiltonian.get_orbitals().get_num_molecular_orbitals() * 2
qubit_hamiltonian = mapper.run(hamiltonian, MajoranaMapping.jordan_wigner(n_spin_orbitals))

# 4) Ground-state energy of the qubit Hamiltonian
solver = create("qubit_hamiltonian_solver")
qubit_energy, ground_state = solver.run(qubit_hamiltonian)
qubit_energy = qubit_energy

This cell takes about 40 secs

In [2]:
from utils.qpe_utils import compute_evolution_time
from qdk_chemistry.data import AlgorithmRef
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
import pandas as pd


def run_trotter(m_precision, trotter_order, target_accuracy=1e-2):
    evolution_time = compute_evolution_time(qubit_hamiltonian, num_bits=m_precision)
    unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "trotter", time=evolution_time, order=trotter_order, power_strategy="rescale", target_accuracy=target_accuracy)
    circuit_mapper = AlgorithmRef("controlled_circuit_mapper", "pauli_sequence")
    circuit_builder = create(
        "qpe_circuit_builder", 
        "qdk_iterative",
        num_bits=m_precision,
        unitary_builder=unitary_builder,
        controlled_circuit_mapper=circuit_mapper,
    )
    state_prep = identity_state_prep(num_qubits=qubit_hamiltonian.num_qubits)
    iqpe_iter_circuits = circuit_builder.run(
        state_preparation=state_prep,
        qubit_hamiltonian=qubit_hamiltonian,
    )
    largest_circuit = iqpe_iter_circuits[0]
    result = largest_circuit.estimate()
    logical_estimate = result["logicalCounts"]

    return logical_estimate, target_accuracy, largest_circuit

rows = []
for m_precision in [6, 8, 10]:
    for trotter_order in [1, 2]:
        logical_estimate, target_accuracy, largest_circuit = run_trotter(m_precision, trotter_order)
        rows.append({
            "m_precision": m_precision,
            "trotter_order": trotter_order,
            "target_accuracy": target_accuracy,
            "numQubits": logical_estimate["numQubits"],
            "rotationCount": logical_estimate["rotationCount"],
            "rotationDepth": logical_estimate["rotationDepth"],
        })

df = pd.DataFrame(rows)
df


,m_precision,trotter_order,target_accuracy,numQubits,rotationCount,rotationDepth
0,6,1,0.01,5,687793,592925
1,6,2,0.01,5,28280,24240
2,8,1,0.01,5,10766395,9281375
3,8,2,0.01,5,222376,190608
4,10,1,0.01,5,173209779,149318775
5,10,2,0.01,5,1785952,1530816


In [ ]:
from qdk.widgets import Circuit

Circuit(largest_circuit.get_qsharp_circuit())